# Binary Classification - No Hypoxia (Normal) or Mild Hypoxia (Suspicious)

For the initial experiments, a binary classification task was defined to distinguish normal CTG recordings from mildly hypoxic (suspicious) recordings. Records labelled as “No Hypoxia (Normal)” were assigned to class 0, while records labelled as “Mild Hypoxia (Suspicious)” were assigned to class 1. This setting focuses on subtle pattern differences rather than overt pathological changes.

This binary classification experiment derives from Expert Annotation data, provided within the original dataset: 
PhysioNet link to dataset - https://physionet.org/content/ctu-uhb-ctgdb/1.0.0/

Paper describing the dataset - https://bmcpregnancychildbirth.biomedcentral.com/track/pdf/10.1186/1471-2393-14-16.pdf

Link to expert evaluation as described in Hruban et al. 2015 - http://people.ciirc.cvut.cz/~spilkjir/data.html

Expert evaluation of the CTG data "Gold Standard" evaluation based on annotation of the signals by 9 expert obstetricians (following FIGO guidelines used in the Czech Republic) including variability/confidence for each signal

## Custom Model for Binary Classification
This part of the project aims to get initial results by asking this question: 
"Given a full CTG record (FHR signal), can we predict whether the fetus ever showed signs of mild hypoxia?"

| Challenge                               | Strategy                                                                 |
|----------------------------------------|--------------------------------------------------------------------------|
| CTG recordings are very long (hours)   | Break long FHR signals into shorter, fixed-length windows                |
| Hypoxia events are episodic            | Use window-based analysis to capture local abnormal patterns              |
| Labels are record-level, not moment-level | Aggregate window-level predictions into a single record-level decision |
| FHR signals are noisy and non-stationary | Use CNNs to learn robust local temporal patterns automatically           |

This model uses **1D Residual Convolutional Neural Network**, which designed for time-series and local temporal dependencies. 

### Residual Block: 
Residual block is a neural-network building unit that learns changes instead of learning the entire signal from scratch. Residual block were invented because of some issues such as vanishing gradient problem and the degradation problem. residual blocks solved this by stabling gradient and with the easy information flow, which led to ResNet (We will be using the pre-Trained ResNet in the next notebook)

x → Conv → BN → ReLU → Conv → BN

|________________________|

            +
            
BN: Batch Normalisation
ReLu: Rectified Linear Unit (Activation function) - this keeps positive values and kills negatives. 

"+" is an element-wise addition between the input and the transformed signal, allowing network to learn residual correction rather than the full mapping. This preserves baseline information and stabilises the training and prevents vanishing gradient problem. 

### Layer by Layer Model Architecture: 
Input: (4800 samples, 1 channel)
Initial Conv Block: Conv1D(64 filters, kernel 7), Batch Normalisation, ReLu. This is to capture basic heart rate patterns. 
Residual Block 1: 64 filters. Conv → Conv → Add → Pool. This is to capture low-level temporal features. 
Residual Block 2: 128 Filters, for mid-level patterns like accelerations or decelerations. 
Residual Block 3: 256 filters , Higher level abstraction. 
Flatten(): preserve the remporaş structure 
Total Depth: ~10 convolutional layers and 1 fully connected classifier

In 3 residual blocks:
| Block   | Filters | Shortcut Conv needed? |
| ------- | ------- | --------------------- |
| Block 1 | 64      | No (64 → 64)        |
| Block 2 | 128     | Yes (64 → 128)      |
| Block 3 | 256     | Yes (128 → 256)     |

Block 1: 2 convs

Block 2: 3 convs
Block 3: 3 convs 

total from residual blocks = 8 conv layers.
Then;

(Initial Conv: 1 conv) + (Residual Blocks = 8 conv) = 9 Convo1D layers.

### Record Level Evaluation
Records are what the labels mean. This model does not trust window predictions only. This answers şf the CTG contain hypoxic episodes at any time during the record, which matches the clinical reality and label definition. 





In [1]:
# Ensure required packages are installed (fix import resolution)
%pip install -q scikit-learn
%pip install -q tensorflow
%pip install -q tensorflow-cpu scikit-learn

# Necessary imports 
import pandas as pd 
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix
from sklearn.utils import class_weight

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow-cpu 2.20.0 requires ml_dtypes<1.0.0,>=0.5.1, but you have ml-dtypes 0.4.1 which is incompatible.
tensorflow-cpu 2.20.0 requires tensorboard~=2.20.0, but you have tensorboard 2.18.0 which is incompatible.

[notice] A new release of pip is available: 24.3.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow-intel 2.18.0 requires ml-dtypes<0.5.0,>=0.4.0, but you have ml-dtypes 0.5.4 which is incompatible.
tensorflow-intel 2.18.0 requires tensorboard<2.19,>=2.18, but you have tensorboard 2.20.0 which is incompatible.

[notice] A new release of pip is available: 24.3.1 -> 25.3
[notice] To update, run: py

Note: you may need to restart the kernel to use updated packages.


In [2]:
'''
Load and prepare labels
'''
labels_df = pd.read_csv("../ExpertAnnotations/CTG_Majority_Vote_Labels_FINAL.csv")
labels_df["binary_label"] = labels_df["Majority_Vote_Label"].map({1: 0, 2: 1})
labels_df = labels_df.dropna(subset=["binary_label"])



In [3]:
'''
Signal Windowing: 
As CNNs require fixed-length input and they are local and temporal, we segment CTG signals into overlapping windows.
These windows helps learn subtle differences in CTG patterns.

Parameters:
Sampling rate: 4 Hz
Window length: 20 minutes
Window size: 20 * 60 * 4 = 4800 samples 20 60 2 = 2400 
Stride: 5 minutes (overlap helps)
'''

FS = 4
WINDOW_MIN = 20
STRIDE_MIN = 5

WINDOW_SIZE = WINDOW_MIN * 60 * FS
STRIDE = STRIDE_MIN * 60 * FS

In [4]:
'''
Window Extraction Function
is _informative_window: filters out uninformative windows based on std and amplitude range
normalize_window: standardizes windows to zero mean and unit variance
'''
def extract_windows(signal, window_size, stride):
    return np.array([
        signal[i:i + window_size]
        for i in range(0, len(signal) - window_size + 1, stride)
    ])

def is_informative_window(w):
    return (
        np.std(w) > 5 and
        (np.max(w) - np.min(w)) > 15
    )

def normalize_window(w):
    return (w - np.mean(w)) / (np.std(w) + 1e-6)


In [5]:
'''
Load FHR Signals from Cleaned CSVs
'''
FHR_DIR = Path("../cleaned_data/clean_csv")

def load_fhr_signal(rec_id):
    path = FHR_DIR / f"{rec_id}.csv"
    if not path.exists():
        return None
    return pd.read_csv(path).values.squeeze()


In [6]:
'''
Extract Windows and Prepare Dataset 
For each record, load FHR signal, extract windows, filter informative ones, normalize, and store with labels.
'''

X, y, rec_ids = [], [], []

for _, row in labels_df.iterrows():
    rec_id = row["rec_id"]
    label = row["binary_label"]

    signal = load_fhr_signal(rec_id)
    if signal is None:
        continue

    windows = extract_windows(signal, WINDOW_SIZE, STRIDE)

    for w in windows:
        if not is_informative_window(w):
            continue
        X.append(normalize_window(w))
        y.append(label)
        rec_ids.append(rec_id)

X = np.array(X)
y = np.array(y)
rec_ids = np.array(rec_ids)

print("Total informative windows:", X.shape[0])

Total informative windows: 4882


In [7]:
'''
Train-Val Split at Record Level:
Split by record, not by window, to avoid data leakage.
'''
unique_recs = np.unique(rec_ids)

train_recs, val_recs = train_test_split(
    unique_recs,
    test_size=0.2,
    stratify=[
        labels_df.loc[labels_df.rec_id == r, "binary_label"].values[0]
        for r in unique_recs
    ]
)

train_mask = np.isin(rec_ids, train_recs)
val_mask = np.isin(rec_ids, val_recs)

X_train, y_train, rec_train = X[train_mask], y[train_mask], rec_ids[train_mask]
X_val, y_val, rec_val = X[val_mask], y[val_mask], rec_ids[val_mask]

In [8]:
# Debugging

print(f"X_train shape: {X_train.shape}")
print(f"X_val shape: {X_val.shape}")
print(f"Class distribution - Train: {np.bincount(y_train)}")
print(f"Class distribution - Val: {np.bincount(y_val)}")

X_train shape: (3923, 4800)
X_val shape: (959, 4800)
Class distribution - Train: [3047  876]
Class distribution - Val: [743 216]


In [17]:
'''Calculate Class Weights to Handle the class Imbalance'''
weights = class_weight.compute_class_weight(
    class_weight="balanced",
    classes=np.array([0, 1]),
    y=y_train
)

class_weight_actual = {0: weights[0], 1: weights[1]}
class_weight_dict = {0: 1.0, 1: 4.2} # Manually set for debugging
print("Class weights:", class_weight_actual)
print("Agressive class weigting:", class_weight_dict)


Class weights: {0: np.float64(0.6437479488021004), 1: np.float64(2.2391552511415527)}
Agressive class weigting: {0: 1.0, 1: 4.2}


In [18]:
'''
Build the Model
'''
def residual_block(x, filters):
    shortcut = x

    x = layers.Conv1D(filters, 3, padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)

    x = layers.Conv1D(filters, 3, padding="same")(x)
    x = layers.BatchNormalization()(x)

    if shortcut.shape[-1] != filters:
        shortcut = layers.Conv1D(filters, 1, padding="same")(shortcut)

    x = layers.Add()([x, shortcut])
    x = layers.Activation("relu")(x)

    return layers.MaxPooling1D(2)(x)


inputs = layers.Input(shape=(WINDOW_SIZE, 1))
x = layers.Conv1D(64, 7, padding="same")(inputs)
x = layers.BatchNormalization()(x)
x = layers.Activation("relu")(x)

x = residual_block(x, 64)
x = residual_block(x, 128)
x = residual_block(x, 256)

x = layers.Flatten()(x)
x = layers.Dense(128, activation="relu")(x)
x = layers.Dropout(0.3)(x)

outputs = layers.Dense(1, activation="sigmoid")(x)

model = models.Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss=tf.keras.losses.BinaryCrossentropy(),
    metrics=[tf.keras.metrics.AUC(name="auc"), tf.keras.metrics.BinaryAccuracy(name="accuracy")]
)

model.summary()

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2       │ (None, 4800, 1)   │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_18 (Conv1D)  │ (None, 4800, 64)  │        512 │ input_layer_2[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 4800, 64)  │        256 │ conv1d_18[0][0]   │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_14       │ (None, 4800, 64)  │          0 │ batch_normalizat… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_19 (Conv1D)  │ (None, 4800, 64)  │     12,352 │ activation_14[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 4800, 64)  │        256 │ conv1d_19[0][0]   │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_15       │ (None, 4800, 64)  │          0 │ batch_normalizat… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_20 (Conv1D)  │ (None, 4800, 64)  │     12,352 │ activation_15[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 4800, 64)  │        256 │ conv1d_20[0][0]   │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_6 (Add)         │ (None, 4800, 64)  │          0 │ batch_normalizat… │
│                     │                   │            │ activation_14[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_16       │ (None, 4800, 64)  │          0 │ add_6[0][0]       │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d_6     │ (None, 2400, 64)  │          0 │ activation_16[0]… │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_21 (Conv1D)  │ (None, 2400, 128) │     24,704 │ max_pooling1d_6[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 2400, 128) │        512 │ conv1d_21[0][0]   │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_17       │ (None, 2400, 128) │          0 │ batch_normalizat… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_22 (Conv1D)  │ (None, 2400, 128) │     49,280 │ activation_17[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 2400, 128) │        512 │ conv1d_22[0][0]   │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_23 (Conv1D)  │ (None, 2400, 128) │      8,320 │ max_pooling1d_6[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_7 (Add)         │ (None, 2400, 128) │          0 │ batch_normalizat

 Total params: 20,100,865 (76.68 MB)

 Trainable params: 20,098,945 (76.67 MB)

 Non-trainable params: 1,920 (7.50 KB)

In [19]:
'''
Train the Model
'''
model.fit(
    X_train[..., np.newaxis],
    y_train,
    epochs=30,
    batch_size=32,
    class_weight=class_weight_dict,
    validation_data=(X_val[..., np.newaxis], y_val),
    
    callbacks=[
    tf.keras.callbacks.EarlyStopping(
        monitor="val_auc",
        mode="max",
        patience=10,
        min_delta=0.001,
        restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_auc',
        factor=0.5,
        patience=5,
        min_lr=1e-6,
        mode='max'
    )
]
)

Epoch 1/30
123/123 ━━━━━━━━━━━━━━━━━━━━ 702s 6s/step - accuracy: 0.4463 - auc: 0.5556 - loss: 1.7478 - val_accuracy: 0.2252 - val_auc: 0.4945 - val_loss: 0.7584 - learning_rate: 1.0000e-04
Epoch 2/30
123/123 ━━━━━━━━━━━━━━━━━━━━ 850s 7s/step - accuracy: 0.5501 - auc: 0.6535 - loss: 1.1146 - val_accuracy: 0.2252 - val_auc: 0.4952 - val_loss: 0.9750 - learning_rate: 1.0000e-04
Epoch 3/30
 71/123 ━━━━━━━━━━━━━━━━━━━━ 2:13 3s/step - accuracy: 0.5011 - auc: 0.7079 - loss: 1.0696

KeyboardInterrupt: 

In [ ]:
''' 
Evaluate the model on validation set
'''
val_preds = model.predict(X_val[..., np.newaxis]).squeeze()

record_scores = {}
for score, rid in zip(val_preds, rec_val):
    record_scores.setdefault(rid, []).append(score)

y_true_rec = []
y_pred_rec = []

for rid, scores in record_scores.items():
    y_pred_rec.append(np.mean(scores))
    y_true_rec.append(
        labels_df.loc[labels_df.rec_id == rid, "binary_label"].values[0]
    )

auc = roc_auc_score(y_true_rec, y_pred_rec)
print("Record-level AUC:", auc)

print(classification_report(
    y_true_rec,
    (np.array(y_pred_rec) > 0.5).astype(int)
))
conf_matrix = confusion_matrix(
    y_true_rec,
    (np.array(y_pred_rec) > 0.5).astype(int)
)


30/30 ━━━━━━━━━━━━━━━━━━━━ 5s 156ms/step
Record-level AUC: 0.6159999999999999
              precision    recall  f1-score   support

           0       0.83      0.93      0.88        85
           1       0.60      0.36      0.45        25

    accuracy                           0.80       110
   macro avg       0.72      0.64      0.66       110
weighted avg       0.78      0.80      0.78       110



In [ ]:
# =========================
# Environment & Imports
# =========================
%pip install -q tensorflow-cpu scikit-learn

import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.utils import class_weight
from sklearn.metrics import roc_auc_score, classification_report

import tensorflow as tf
from tensorflow.keras import layers, models


# =========================
# Load Labels
# =========================
labels_df = pd.read_csv("../ExpertAnnotations/CTG_Majority_Vote_Labels_FINAL.csv")
labels_df["binary_label"] = labels_df["Majority_Vote_Label"].map({1: 0, 2: 1})
labels_df = labels_df.dropna(subset=["binary_label"])


# =========================
# Parameters
# =========================
FS = 4
WINDOW_MIN = 20
STRIDE_MIN = 5

WINDOW_SIZE = WINDOW_MIN * 60 * FS
STRIDE = STRIDE_MIN * 60 * FS


# =========================
# Utility Functions
# =========================
def extract_windows(signal, window_size, stride):
    return np.array([
        signal[i:i + window_size]
        for i in range(0, len(signal) - window_size + 1, stride)
    ])

def is_informative_window(w):
    return (
        np.std(w) > 5 and
        (np.max(w) - np.min(w)) > 15
    )

def normalize_window(w):
    return (w - np.mean(w)) / (np.std(w) + 1e-6)


# =========================
# Load Signals
# =========================
FHR_DIR = Path("../cleaned_data/clean_csv")

def load_fhr_signal(rec_id):
    path = FHR_DIR / f"{rec_id}.csv"
    if not path.exists():
        return None
    return pd.read_csv(path).values.squeeze()


# =========================
# Build Dataset (FILTERED)
# =========================
X, y, rec_ids = [], [], []

for _, row in labels_df.iterrows():
    rec_id = row["rec_id"]
    label = row["binary_label"]

    signal = load_fhr_signal(rec_id)
    if signal is None:
        continue

    windows = extract_windows(signal, WINDOW_SIZE, STRIDE)

    for w in windows:
        if not is_informative_window(w):
            continue
        X.append(normalize_window(w))
        y.append(label)
        rec_ids.append(rec_id)

X = np.array(X)
y = np.array(y)
rec_ids = np.array(rec_ids)

print("Total informative windows:", X.shape[0])


# =========================
# Record-Level Split
# =========================
unique_recs = np.unique(rec_ids)

train_recs, val_recs = train_test_split(
    unique_recs,
    test_size=0.2,
    random_state=42,
    stratify=[
        labels_df.loc[labels_df.rec_id == r, "binary_label"].values[0]
        for r in unique_recs
    ]
)

train_mask = np.isin(rec_ids, train_recs)
val_mask = np.isin(rec_ids, val_recs)

X_train, y_train, rec_train = X[train_mask], y[train_mask], rec_ids[train_mask]
X_val, y_val, rec_val = X[val_mask], y[val_mask], rec_ids[val_mask]


# =========================
# Class Weights
# =========================
weights = class_weight.compute_class_weight(
    class_weight="balanced",
    classes=np.array([0, 1]),
    y=y_train
)

class_weight_dict = {0: weights[0], 1: weights[1]}
print("Class weights:", class_weight_dict)


# =========================
# Model Definition
# =========================
def residual_block(x, filters):
    shortcut = x

    x = layers.Conv1D(filters, 3, padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)

    x = layers.Conv1D(filters, 3, padding="same")(x)
    x = layers.BatchNormalization()(x)

    if shortcut.shape[-1] != filters:
        shortcut = layers.Conv1D(filters, 1, padding="same")(shortcut)

    x = layers.Add()([x, shortcut])
    x = layers.Activation("relu")(x)

    return layers.MaxPooling1D(2)(x)


inputs = layers.Input(shape=(WINDOW_SIZE, 1))
x = layers.Conv1D(64, 7, padding="same")(inputs)
x = layers.BatchNormalization()(x)
x = layers.Activation("relu")(x)

x = residual_block(x, 64)
x = residual_block(x, 128)
x = residual_block(x, 256)

x = layers.Flatten()(x)
x = layers.Dense(128, activation="relu")(x)
x = layers.Dropout(0.3)(x)

outputs = layers.Dense(1, activation="sigmoid")(x)

model = models.Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(3e-4),
    loss="binary_crossentropy",
    metrics=[tf.keras.metrics.AUC(name="auc")]
)

model.summary()


# =========================
# Train
# =========================
model.fit(
    X_train[..., np.newaxis],
    y_train,
    epochs=30,
    batch_size=32,
    class_weight=class_weight_dict,
    validation_data=(X_val[..., np.newaxis], y_val),
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            monitor="val_auc",
            mode="max",
            patience=8,
            restore_best_weights=True
        )
    ]
)


# =========================
# RECORD-LEVEL EVALUATION
# =========================
val_preds = model.predict(X_val[..., np.newaxis]).squeeze()

record_scores = {}
for score, rid in zip(val_preds, rec_val):
    record_scores.setdefault(rid, []).append(score)

y_true_rec = []
y_pred_rec = []

for rid, scores in record_scores.items():
    y_pred_rec.append(np.mean(scores))
    y_true_rec.append(
        labels_df.loc[labels_df.rec_id == rid, "binary_label"].values[0]
    )

auc = roc_auc_score(y_true_rec, y_pred_rec)
print("Record-level AUC:", auc)

print(classification_report(
    y_true_rec,
    (np.array(y_pred_rec) > 0.5).astype(int)
))


ERROR: Could not install packages due to an OSError: [WinError 5] Erişim engellendi: 'c:\\Users\\ZEYNEP\\AppData\\Local\\Programs\\Python\\Python310\\Lib\\site-packages\\tensorflow\\compiler\\mlir\\lite\\python\\_pywrap_converter_api.pyd'
Consider using the `--user` option or check the permissions.


[notice] A new release of pip is available: 24.3.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
Total informative windows: 4882
Class weights: {0: np.float64(0.6452513966480447), 1: np.float64(2.2211538461538463)}


Model: "functional_10"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_6       │ (None, 4800, 1)   │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_45 (Conv1D)  │ (None, 4800, 64)  │        512 │ input_layer_6[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 4800, 64)  │        256 │ conv1d_45[0][0]   │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_30       │ (None, 4800, 64)  │          0 │ batch_normalizat… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_46 (Conv1D)  │ (None, 4800, 64)  │     12,352 │ activation_30[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 4800, 64)  │        256 │ conv1d_46[0][0]   │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_31       │ (None, 4800, 64)  │          0 │ batch_normalizat… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_47 (Conv1D)  │ (None, 4800, 64)  │     12,352 │ activation_31[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 4800, 64)  │        256 │ conv1d_47[0][0]   │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_12 (Add)        │ (None, 4800, 64)  │          0 │ batch_normalizat… │
│                     │                   │            │ activation_30[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_32       │ (None, 4800, 64)  │          0 │ add_12[0][0]      │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d_20    │ (None, 2400, 64)  │          0 │ activation_32[0]… │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_48 (Conv1D)  │ (None, 2400, 128) │     24,704 │ max_pooling1d_20… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 2400, 128) │        512 │ conv1d_48[0][0]   │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_33       │ (None, 2400, 128) │          0 │ batch_normalizat… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_49 (Conv1D)  │ (None, 2400, 128) │     49,280 │ activation_33[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 2400, 128) │        512 │ conv1d_49[0][0]   │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_50 (Conv1D)  │ (None, 2400, 128) │      8,320 │ max_pooling1d_20… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_13 (Add)        │ (None, 2400, 128) │          0 │ batch_normalizat

 Total params: 20,100,865 (76.68 MB)

 Trainable params: 20,098,945 (76.67 MB)

 Non-trainable params: 1,920 (7.50 KB)

Epoch 1/30
123/123 ━━━━━━━━━━━━━━━━━━━━ 111s 877ms/step - auc: 0.5010 - loss: 5.0712 - val_auc: 0.5000 - val_loss: 0.6932
Epoch 2/30
123/123 ━━━━━━━━━━━━━━━━━━━━ 113s 921ms/step - auc: 0.4986 - loss: 0.6905 - val_auc: 0.5000 - val_loss: 0.6932
Epoch 3/30
123/123 ━━━━━━━━━━━━━━━━━━━━ 129s 1s/step - auc: 0.5211 - loss: 0.6980 - val_auc: 0.4983 - val_loss: 0.6938
Epoch 4/30
123/123 ━━━━━━━━━━━━━━━━━━━━ 126s 1s/step - auc: 0.5403 - loss: 0.6899 - val_auc: 0.5163 - val_loss: 0.7003
Epoch 5/30
123/123 ━━━━━━━━━━━━━━━━━━━━ 141s 1s/step - auc: 0.5511 - loss: 0.6776 - val_auc: 0.5324 - val_loss: 0.6843
Epoch 6/30
123/123 ━━━━━━━━━━━━━━━━━━━━ 119s 968ms/step - auc: 0.6509 - loss: 0.6507 - val_auc: 0.5291 - val_loss: 0.6929
Epoch 7/30
123/123 ━━━━━━━━━━━━━━━━━━━━ 114s 930ms/step - auc: 0.6621 - loss: 0.6361 - val_auc: 0.5881 - val_loss: 0.6495
Epoch 8/30
123/123 ━━━━━━━━━━━━━━━━━━━━ 115s 939ms/step - auc: 0.7176 - loss: 0.6086 - val_auc: 0.6116 - val_loss: 0.5832
Epoch 9/30
123/123 ━━━━━━━━━━━━━━

In [17]:
from sklearn.metrics import confusion_matrix, classification_report

# Convert probabilities to binary predictions
y_pred_binary = (np.array(y_pred_rec) > 0.5).astype(int)

# Confusion matrix
cm = confusion_matrix(y_true_rec, y_pred_binary)

print("Record-level Confusion Matrix:"),
print(cm)

print("\nClassification Report:")
print(classification_report(y_true_rec, y_pred_binary, target_names=["Normal", "Mild Hypoxia"]))


Record-level Confusion Matrix:
[[64 21]
 [ 9 16]]

Classification Report:
              precision    recall  f1-score   support

      Normal       0.88      0.75      0.81        85
Mild Hypoxia       0.43      0.64      0.52        25

    accuracy                           0.73       110
   macro avg       0.65      0.70      0.66       110
weighted avg       0.78      0.73      0.74       110

